# Append simulations to an existing SDL experiment

Adds **new simulations** to an experiment that is **already created** (no new experiment, no authors needed).

**Setup -> Login -> pick experiment + version -> pick folder -> Append.**

- Point to a folder with only the **new** simulations: `parfiles/` (the new `.txt`) + `sim/` (their `.nc`/`_ts.nc`, matching each parfile's NetCDF prefix).
- **`reuse_scenario` marcado** = añadir a una versión **existente** (no re-sube la batimetría; enlaza al scenario ya presente). Deja la **versión existente** seleccionada.
- Para una **versión nueva**: **desmarca** `reuse_scenario` e incluye `bathy/` en la carpeta (se re-sube la batimetría para esa versión).


In [ ]:
import os, sys, glob, yaml
for p in ("~/HySEALab/Notebooks/01_sdl_publish", "~/SDL", "~/HySEALab/Notebooks", "."):
    pp = os.path.expanduser(p)
    if os.path.exists(os.path.join(pp, "sdl_lib.py")):
        sys.path.insert(0, pp); break
import importlib, sdl_lib; importlib.reload(sdl_lib)
import ipywidgets as W
from IPython.display import display
STATE = {}
print("sdl_lib loaded from:", os.path.dirname(sdl_lib.__file__))

In [ ]:
srv = W.Dropdown(options=[("prod", "prod"), ("dev", "dev")], value="prod", description="Server:")
u   = W.Text(description="SDL user:", layout=W.Layout(width="380px"))
pw  = W.Password(description="Password:", layout=W.Layout(width="380px"))
b   = W.Button(description="Login", button_style="primary")
o   = W.Output()

def _login(_):
    with o:
        o.clear_output()
        base = sdl_lib.SERVERS.get(srv.value, sdl_lib.SERVERS["prod"])
        try:
            tok = sdl_lib.get_auth_token(base, u.value, pw.value)
            STATE["token"] = tok; STATE["headers"] = sdl_lib.auth_headers(tok); STATE["base"] = base
            print("Login OK on", base)
        except Exception as e:
            print("Login failed:", type(e).__name__, e)

b.on_click(_login)
display(W.VBox([srv, u, pw, b, o]))

In [ ]:
def _as_list(x):
    if isinstance(x, dict):
        for k in ("content", "res", "items", "data", "experiments"):
            if isinstance(x.get(k), list):
                return x[k]
        return [x]
    return x if isinstance(x, list) else []

exp_dd   = W.Dropdown(options=[], description="Experiment:", layout=W.Layout(width="560px"))
load_exp = W.Button(description="Load experiments", button_style="info", layout=W.Layout(width="180px"))
ver_dd   = W.Dropdown(options=[], description="Version:", layout=W.Layout(width="300px"))
new_ver  = W.Text(description="or new:", placeholder="e.g. 1.1.0", layout=W.Layout(width="260px"))
reuse    = W.Checkbox(value=True, description="reuse_scenario (no re-subir batimetria)")
scen     = W.Text(value="event_data", description="scenario:", layout=W.Layout(width="300px"))
base     = W.Text(value="~", description="Search in:", layout=W.Layout(width="430px"))
find     = W.Button(description="Find folders", layout=W.Layout(width="160px"))
folder   = W.Dropdown(options=[], description="Folder:", layout=W.Layout(width="720px"))
manual   = W.Text(value="", description="or path:", placeholder="folder with parfiles/ + sim/ (new sims)", layout=W.Layout(width="720px"))
selout   = W.Output()
out      = W.Output()

def _load_exps(_):
    with selout:
        selout.clear_output()
        if not STATE.get("token"):
            print("Login first."); return
        try:
            raw = _as_list(sdl_lib.list_experiments(STATE["base"], STATE["headers"]))
        except Exception as e:
            print("Error listing experiments:", e); return
        opts = []
        for it in raw:
            if not isinstance(it, dict):
                continue
            i = it.get("id"); t = it.get("title") or it.get("name") or ""
            if i is not None:
                opts.append(("%s - %s" % (i, str(t)[:60]), i))
        exp_dd.options = sorted(opts, key=lambda o: (o[1] if isinstance(o[1], int) else 0), reverse=True)
        print("experiments:", len(opts))
load_exp.on_click(_load_exps)

def _load_vers(_=None):
    with selout:
        if not STATE.get("token") or exp_dd.value is None:
            return
        try:
            vs = _as_list(sdl_lib.list_versions(STATE["base"], STATE["headers"], exp_dd.value))
        except Exception as e:
            print("Error versions:", e); return
        names = [(v.get("version") if isinstance(v, dict) else str(v)) for v in vs]
        ver_dd.options = [x for x in names if x]
exp_dd.observe(lambda ch: _load_vers(), names="value")

def _find(_):
    with selout:
        root = os.path.expanduser(base.value.strip() or "~")
        hits = sorted({os.path.dirname(p) for p in glob.glob(os.path.join(root, "**", "parfiles"), recursive=True)})
        folder.options = hits
        print("folders with parfiles/:", len(hits))
find.on_click(_find)

def _chosen_folder():
    return os.path.expanduser(manual.value.strip()) if manual.value.strip() else (folder.value or "")

def _expected(root):
    p = os.path.join(root, "parfiles"); s = os.path.join(root, "sim")
    np = len([f for f in os.listdir(p) if f.lower().endswith((".txt", ".par"))]) if os.path.isdir(p) else 0
    ns = len(os.listdir(s)) if os.path.isdir(s) else 0
    return np + ns

btn = W.Button(description="Append simulations", button_style="primary", layout=W.Layout(width="240px"))

def _append(_):
    with out:
        out.clear_output()
        if not STATE.get("token"):
            print("Login first."); return
        exp_id = exp_dd.value
        if exp_id is None:
            print("Pick an experiment (Load experiments)."); return
        ver = (new_ver.value.strip() or ver_dd.value or "").strip()
        if not ver:
            print("Set a version (existing, or type a new one)."); return
        root = _chosen_folder()
        if not root or not os.path.isdir(os.path.join(root, "parfiles")) or not os.path.isdir(os.path.join(root, "sim")):
            print("The folder must contain parfiles/ and sim/ with the NEW simulations."); return
        server = "dev" if STATE["base"].endswith("dev.hpc.cineca.it") else "prod"
        manifest = {
            "sdl": {"server": server},
            "experiment": {"mode": "existing", "id": exp_id},
            "version": {"name": ver, "create_if_missing": True},
            "publication": {"visibility": "private"},
            "project": {"root": ".", "reuse_scenario": bool(reuse.value),
                        "scenario_name": scen.value.strip() or "event_data"},
        }
        mf = os.path.join(root, "sdl_manifest.append.yml")
        with open(mf, "w") as f:
            yaml.safe_dump(manifest, f, sort_keys=False, allow_unicode=True)
        print("Manifest ->", mf)
        print(yaml.safe_dump(manifest, sort_keys=False))
        tot = _expected(root)
        bar = W.IntProgress(value=0, min=0, max=max(tot, 1), description="exp %s" % exp_id,
                            bar_style="info", layout=W.Layout(width="540px"))
        lab = W.Label(value="0/%d files" % tot)
        display(W.VBox([bar, lab]))
        def _log(msg, _b=bar, _l=lab, _t=tot):
            print(msg)
            parts = str(msg).strip().split(None, 1)
            if len(parts) == 2 and parts[0] == "✅" and " " not in parts[1]:
                if _t and _b.value < _t: _b.value += 1
                _l.value = ("%d/%d files - %s" % (_b.value, _t, parts[1][:60])) if _t else parts[1][:70]
        try:
            r = sdl_lib.publish_from_manifest(mf, token=STATE["token"], headers=STATE["headers"],
                                              base_url=STATE["base"], log=_log)
            bar.value = bar.max; bar.bar_style = "success"
            lab.value = "DONE - exp %s v%s" % (r.get("experiment_id"), r.get("version"))
            print("OK: simulations added to experiment %s version %s : %s/app/"
                  % (r.get("experiment_id"), r.get("version"), r.get("base_url")))
        except Exception as e:
            bar.bar_style = "danger"
            print("ERROR:", type(e).__name__, e)
btn.on_click(_append)

display(W.VBox([
    W.HBox([load_exp]), exp_dd, W.HBox([ver_dd, new_ver]),
    W.HBox([reuse, scen]),
    W.HBox([base, find]), folder, manual, selout,
    btn, out,
]))